In [10]:
import os
import matplotlib.pyplot as plt
import numpy as np
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_files

# Load dữ liệu từ thư mục

In [11]:
INPUT = 'data/news_vnexpress'

listdir() trả về về 1 list là tên các thư mục và file nằm trong tham số mà ta truyền vào. Tham số ta truyền vào là 1 đường dẫn tương đối như ta làm ở trên do data nằm cùng vị trí với file code nên có thể bỏ qua vị trí trước hoặc đường dẫn tuyệt đối. Đặc biệt tham số truyền vào không được là đường dẫn tới 1 file! 

isdir() trả về True khi tham số truyền vào của ta là 1 thư mục chứ không phải 1 file .txt, .jpg hay 1 file không tồn tại

.path.join() thường có 2 tham số truyền vào là đường dẫn tới file gốc INPUT và tên thư mục con trong file gốc đó thì ta thu được đường dẫn tới thư mục con. Đặc biệt là do ta dùng listdir với thư mục con nên phải kiểm tra thư mục con đó có thực sự là thư mục không, nếu không code sẽ báo lỗi ! 
    
    Thực ra bản chất .path.join là có thể có nhiều tham số truyền vào là các xâu và sẽ thực hiện ghép các xâu đó lại để tạo thành 1 đường dẫn hoàn chỉnh

In [12]:
print('Các nhãn và số văn bản tương ứng trong dữ liệu')
print('----------------------------------------------')
n = 0
for label in os.listdir(INPUT):
    label_path = os.path.join(INPUT,label)
    if os.path.isdir(label_path):
        num_files = len(os.listdir(label_path))
        print(label, num_files)
        n += num_files
print('----------------------------------------------')
print("Tổng số văn bản:",n)

Các nhãn và số văn bản tương ứng trong dữ liệu
----------------------------------------------
doi-song 120
du-lich 54
phap-luat 59
the-thao 173
thoi-su 59
suc-khoe 162
giai-tri 201
giao-duc 105
kinh-doanh 262
khoa-hoc 144
----------------------------------------------
Tổng số văn bản: 1339


Vì sao ta lại dùng load_files trong sk-learn thay vì 1 thư viện nào đó của python? Câu trả lời là ta có thể biến đổi dữ liệu đó sang kiểu bunch như các dataset có sẵn mà ta tìm hiểu ở phần tutorial nên vô cùng tiện lợi

    Ta sắp xếp các văn bản thuộc cùng 1 chủ đề vào 1 folder. Khi đó tên folder sẽ là nhãn và dữ liệu X là các văn bản trong các folder ! Câu lệnh load_files() sẽ thực hiện tự động việc xác định data = các văn bản và nhãn là têm các folder tổng hợp lại về 1 bảng kiểu bunch !

Data lúc này đang là 1 xâu: cụ thể mỗi mẫu là nội dung của 1 file txt

In [13]:
data_train = load_files(container_path=INPUT,encoding='utf-8')
print('mapping:')
for i in range(len(data_train.target_names)):
    print(data_train.target_names[i],'-',i)
print('--------------------------')
print(data_train.filenames[0:1])
print(data_train.data[0:1])
print(data_train.target[0:1])


mapping:
doi-song - 0
du-lich - 1
giai-tri - 2
giao-duc - 3
khoa-hoc - 4
kinh-doanh - 5
phap-luat - 6
suc-khoe - 7
the-thao - 8
thoi-su - 9
--------------------------
['data/news_vnexpress/khoa-hoc/00133.txt']
['Mời độc giả đặt câu hỏi tại đây\n']
[4]


# Chuyển dữ liệu dạng text về ma trận bằng TF_IDF

Ta lưu ý về stopwords như sau: ở tiếng anh thì stopwords có thể là 1 cụm từ có dấu cách kiểu 'new york' (stopword là các từ không được đưa vào trong từ điển khi thực hiện mã hoá bag-of-words CounterVectorizer() hay TF-IDF TfidfVectorizer() ) nhưng tiếng việt thì 1 cụm từ nhiều tiếng phải được viết liền theo quy ước của thư viện pyvi và underthesea, cách bởi dấu '_' gạch dưới. Do vậy file stopwords của ta với mỗi stopwords viết trên 1 dòng và là 1 cụm từ cách bình thuờng (chưa được chuẩn hoá), ta sẽ cần chuẩn hoá nó

Sử dụng with open as giúp tự động đóng lại file mà không cần câu lệnh close(). Phương thức readlines() của 1 file giúp đọc từng dòng như 1 xâu (xâu kết thúc bằng '\n') và trả về 1 list các xâu đó. 

Tiếp ta cần thực hiện việc xoá dấu '\n' ở cuối và với mỗi xâu thì ta xoá khoảng trắng và thay bằng '_'. Việc xoá dấu '\n' ở cuối có thể dùng hàm strip() (xoá khoảng trắng đầu và cuối xâu) còn việc thay khoảng trắng ở giữa thì dùng replace(). Lưu ý là cả 2 hàm đều trả về xâu nên ta có 1 cú pháp ngắn gọn

In [14]:
# load dữ liệu các stopwords 
with open('data/vietnamese-stopwords.txt') as f:
    stopwords = f.readlines()
for i in range(len(stopwords)):
    stopwords[i] = stopwords[i].strip().replace(' ','_')
print("Số lượng stopwords:",len(stopwords))
print(stopwords[0:10])

# Chuyển hoá dữ liệu text về dạng vector TF 
#     - loại bỏ từ dừng
#     - sinh từ điển
tf = TfidfVectorizer(stop_words=stopwords)
X = tf.fit_transform(data_train.data)
y = data_train.target


Số lượng stopwords: 2063
['a_lô', 'a_ha', 'ai', 'ai_ai', 'ai_nấy', 'ai_đó', 'alô', 'amen', 'anh', 'anh_ấy']


In [15]:
X.shape, y.shape

((1339, 12796), (1339,))

In [16]:
print(X[100].toarray())
print(y[100])

[[0.         0.         0.         ... 0.         0.14048828 0.        ]]
5


Làm thế nào để xem văn bản thứ 101 có bao nhiêu từ khác biệt?
X[100].toarray() != 0 trả về 1 mảng 2 chiều có giá trị booleans. Lần sum() đầu tiên cộng các giá trị 0,1 theo cột nhưng có thể hiểu là đưa về mảng 1 chiều. Lần sum() tiếp theo mới cho ta được số từ khác biệt trong văn bản thứ 101

In [17]:
sum(sum(X[100].toarray() != 0))

np.int64(289)

In [18]:
print(X[100])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 289 stored elements and shape (1, 12796)>
  Coords	Values
  (0, 11017)	0.07753890962780573
  (0, 10198)	0.021696616505624673
  (0, 8706)	0.020613744333044446
  (0, 7475)	0.04772540885069628
  (0, 6088)	0.015908469616898763
  (0, 5142)	0.01701734986816564
  (0, 81)	0.015211595534715629
  (0, 11908)	0.03206234356763185
  (0, 10835)	0.012384285972547808
  (0, 11401)	0.015299319965476526
  (0, 6701)	0.030423191069431258
  (0, 10751)	0.048075959422409754
  (0, 8988)	0.06978862139111384
  (0, 8587)	0.023855951537725493
  (0, 4348)	0.013928319765567743
  (0, 4411)	0.042670304614415976
  (0, 9376)	0.0275475296248762
  (0, 11492)	0.020575343700858305
  (0, 5193)	0.07325927523975806
  (0, 8704)	0.07084119562980157
  (0, 12584)	0.03876774373554222
  (0, 3215)	0.03857618758551902
  (0, 11314)	0.01431451989084914
  (0, 3665)	0.030388370645980864
  (0, 4301)	0.02191211183194475
  :	:
  (0, 6862)	0.04644575549064682
  (0, 11455)	0.15351093